In [4]:
from tensorflow.keras.models import load_model
from tensorflow.keras import layers
import time
import numpy as np
import psutil
import os
import platform
import tensorflow as tf
import warnings
import logging

# suppress verbose TF/Keras logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# ================================================================
#  HELPERS
# ================================================================
def _dtype_itemsize(dtype):
    """Works for both string dtypes (Keras 3) and numpy dtypes."""
    if isinstance(dtype, str):
        return np.dtype(dtype).itemsize
    return dtype.itemsize


def count_parameters(model):
    trainable     = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    non_trainable = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))
    return {
        'trainable'    : trainable,
        'non_trainable': non_trainable,
        'total'        : trainable + non_trainable,
        'trainable_M'  : round(trainable / 1e6, 4),
        'total_M'      : round((trainable + non_trainable) / 1e6, 4),
    }


def get_flops(model, input_shape=(224, 224, 3)):
    """
    Compute FLOPs silently — suppresses the verbose profile report.
    Returns raw FLOPs integer or None on failure.
    """
    try:
        from tensorflow.python.framework.convert_to_constants import \
            convert_variables_to_constants_v2

        dummy  = tf.ones((1, *input_shape))
        cf     = tf.function(model).get_concrete_function(dummy)
        frozen = convert_variables_to_constants_v2(cf)
        graph  = frozen.graph

        # redirect stdout to suppress the profile ASCII table
        import io, sys
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()

        with graph.as_default():
            run_meta = tf.compat.v1.RunMetadata()
            opts = (tf.compat.v1.profiler.ProfileOptionBuilder
                    .float_operation())
            opts['output'] = 'none'           # ← suppresses stdout dump
            prof = tf.compat.v1.profiler.profile(
                graph=graph, run_meta=run_meta,
                cmd='op', options=opts)

        sys.stdout = old_stdout
        return prof.total_float_ops

    except Exception as e:
        sys.stdout = old_stdout if 'old_stdout' in dir() else sys.stdout
        print(f"    ⚠️  FLOPs computation failed: {e}")
        return None


def measure_memory(model, input_shape=(224, 224, 3)):
    process    = psutil.Process(os.getpid())
    ram_before = process.memory_info().rss / 1024 ** 2

    dummy = np.random.rand(1, *input_shape).astype(np.float32)
    _     = model.predict(dummy, verbose=0)

    ram_after = process.memory_info().rss / 1024 ** 2

    # ── fix: handle string dtype (Keras 3) ──────────────────
    param_bytes = sum(
        np.prod(w.shape) * _dtype_itemsize(w.dtype)
        for w in model.weights
    )
    weight_mb = param_bytes / 1024 ** 2

    # disk size
    tmp_path = '/tmp/_model_deploy_check.keras'
    model.save(tmp_path)
    disk_mb = os.path.getsize(tmp_path) / 1024 ** 2
    os.remove(tmp_path)

    return {
        'weight_size_mb': round(weight_mb, 3),
        'disk_size_mb'  : round(disk_mb, 3),
        'ram_before_mb' : round(ram_before, 2),
        'ram_after_mb'  : round(ram_after, 2),
        'ram_delta_mb'  : round(ram_after - ram_before, 2),
    }


def measure_latency(model, input_shape=(224, 224, 3),
                    batch_size=1, n_warmup=20, n_runs=200):
    dummy = np.random.rand(batch_size, *input_shape).astype(np.float32)

    for _ in range(n_warmup):
        model.predict(dummy, verbose=0)

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.predict(dummy, verbose=0)
        times.append((time.perf_counter() - t0) * 1000)   # ms

    times        = np.array(times)
    per_image_ms = times / batch_size
    return {
        'batch_size'       : batch_size,
        'mean_ms'          : float(np.mean(per_image_ms)),
        'std_ms'           : float(np.std(per_image_ms)),
        'median_ms'        : float(np.median(per_image_ms)),
        'p95_ms'           : float(np.percentile(per_image_ms, 95)),
        'p99_ms'           : float(np.percentile(per_image_ms, 99)),
        'throughput_img_s' : float(batch_size * 1000 / np.mean(times)),
    }


def get_hardware_info():
    gpus = tf.config.list_physical_devices('GPU')
    info = {
        'platform'    : platform.platform(),
        'processor'   : platform.processor(),
        'python'      : platform.python_version(),
        'tensorflow'  : tf.__version__,
        'cpu_cores'   : psutil.cpu_count(logical=False),
        'cpu_threads' : psutil.cpu_count(logical=True),
        'ram_total_gb': round(psutil.virtual_memory().total / 1024**3, 2),
        'gpu_devices' : [g.name for g in gpus] if gpus else ['CPU only'],
    }
    if gpus:
        try:
            det = tf.config.experimental.get_device_details(gpus[0])
            info['gpu_name'] = det.get('device_name', 'N/A')
        except Exception:
            pass
    return info


# ================================================================
#  MAIN
# ================================================================
def full_deployment_profile(model, input_shape=(224, 224, 3),
                             batch_sizes=(1, 8, 16, 32)):

    SEP = "=" * 62

    print(f"\n{SEP}")
    print("  DEPLOYMENT METRICS PROFILE")
    print(SEP)

    # ── hardware ────────────────────────────────────────────
    hw = get_hardware_info()
    print("\n  [Hardware]")
    for k, v in hw.items():
        print(f"    {k:<20}: {v}")

    # ── parameters ──────────────────────────────────────────
    params = count_parameters(model)
    print(f"\n  [Parameters]")
    print(f"    Trainable     : {params['trainable']:>12,}  ({params['trainable_M']} M)")
    print(f"    Non-trainable : {params['non_trainable']:>12,}")
    print(f"    Total         : {params['total']:>12,}  ({params['total_M']} M)")

    # ── FLOPs ───────────────────────────────────────────────
    print(f"\n  [FLOPs]")
    flops_raw = get_flops(model, input_shape)
    if flops_raw:
        print(f"    Total FLOPs   : {flops_raw:>15,}")
        print(f"    MFLOPs        : {flops_raw/1e6:>15.2f}")
        print(f"    GFLOPs        : {flops_raw/1e9:>15.4f}")
    else:
        print("    Could not compute FLOPs.")

    # ── memory ──────────────────────────────────────────────
    print(f"\n  [Memory & Model Size]")
    mem = measure_memory(model, input_shape)
    print(f"    Weight size   : {mem['weight_size_mb']:>8.3f} MB")
    print(f"    Disk size     : {mem['disk_size_mb']:>8.3f} MB")
    print(f"    RAM (before)  : {mem['ram_before_mb']:>8.2f} MB")
    print(f"    RAM (after)   : {mem['ram_after_mb']:>8.2f} MB")
    print(f"    RAM delta     : {mem['ram_delta_mb']:>8.2f} MB")

    # ── latency & throughput ─────────────────────────────────
    print(f"\n  [Latency & Throughput]  ({200} timed runs per batch)")
    print(f"  {'Batch':>6} | {'Mean ms':>9} | {'Std ms':>7} | "
          f"{'P95 ms':>8} | {'P99 ms':>8} | {'img/s':>10}")
    print(f"  {'-'*6}-+-{'-'*9}-+-{'-'*7}-+-{'-'*8}-+-{'-'*8}-+-{'-'*10}")

    latency_results = {}
    for bs in batch_sizes:
        r = measure_latency(model, input_shape, batch_size=bs)
        latency_results[bs] = r
        print(f"  {bs:>6} | {r['mean_ms']:>9.3f} | {r['std_ms']:>7.3f} | "
              f"{r['p95_ms']:>8.3f} | {r['p99_ms']:>8.3f} | "
              f"{r['throughput_img_s']:>10.1f}")

    r1 = latency_results[1]
    print(f"\n  [Single-Image Summary  (batch=1)]")
    print(f"    Mean latency  : {r1['mean_ms']:.3f} ± {r1['std_ms']:.3f} ms")
    print(f"    Median        : {r1['median_ms']:.3f} ms")
    print(f"    95th pct      : {r1['p95_ms']:.3f} ms")
    print(f"    Throughput    : {r1['throughput_img_s']:.1f} images/sec")
    print(f"\n{SEP}\n")

    return dict(hardware=hw, params=params, memory=mem,
                latency=latency_results,
                flops_raw=flops_raw or 'N/A')

In [5]:
# Serializable functions 
# ---------------------------
# Squeeze-and-Excitation
# ---------------------------
@tf.keras.utils.register_keras_serializable()
class SEBlock(layers.Layer):
    def __init__(self, reduction=8, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        self.gap = layers.GlobalAveragePooling2D()
        self.fc1 = layers.Dense(C // self.reduction, activation="relu")
        self.fc2 = layers.Dense(C, activation="sigmoid")

    def call(self, x):
        s = self.gap(x)
        s = self.fc1(s)
        s = self.fc2(s)
        s = tf.reshape(s, (-1, 1, 1, x.shape[-1]))
        return x * s


# ---------------------------
# CoordAttention
# ---------------------------
@tf.keras.utils.register_keras_serializable()
class CoordAttention(layers.Layer):
    def __init__(self, reduction=32, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        mip = max(8, C // self.reduction)
        self.conv1 = layers.Conv2D(mip, 1, activation="relu")
        self.conv_h = layers.Conv2D(C, 1)
        self.conv_w = layers.Conv2D(C, 1)

    def call(self, x):
        h = tf.reduce_mean(x, axis=2, keepdims=True)
        w = tf.reduce_mean(x, axis=1, keepdims=True)
        w = tf.transpose(w, [0, 2, 1, 3])

        y = tf.concat([h, w], axis=1)
        y = self.conv1(y)

        h, w = tf.split(y, [tf.shape(h)[1], tf.shape(w)[1]], axis=1)
        w = tf.transpose(w, [0, 2, 1, 3])

        ah = tf.sigmoid(self.conv_h(h))
        aw = tf.sigmoid(self.conv_w(w))
        return x * ah * aw

In [6]:
model = load_model('Proposed_Model.keras')  
results = full_deployment_profile(
    model,
    input_shape = (224, 224, 3), 
    batch_sizes = (1, 8, 16, 32),
)


  DEPLOYMENT METRICS PROFILE

  [Hardware]
    platform            : macOS-26.3-arm64-arm-64bit
    processor           : arm
    python              : 3.11.14
    tensorflow          : 2.16.2
    cpu_cores           : 10
    cpu_threads         : 10
    ram_total_gb        : 16.0
    gpu_devices         : ['/physical_device:GPU:0']
    gpu_name            : METAL

  [Parameters]
    Trainable     :    1,722,281  (1.7223 M)
    Non-trainable :        5,088
    Total         :    1,727,369  (1.7274 M)

  [FLOPs]
    Total FLOPs   :   2,395,853,977
    MFLOPs        :         2395.85
    GFLOPs        :          2.3959

  [Memory & Model Size]
    Weight size   :    6.589 MB
    Disk size     :   20.251 MB
    RAM (before)  :   736.45 MB
    RAM (after)   :  1288.86 MB
    RAM delta     :   552.41 MB

  [Latency & Throughput]  (200 timed runs per batch)
   Batch |   Mean ms |  Std ms |   P95 ms |   P99 ms |      img/s
  -------+-----------+---------+----------+----------+-----------
   